# KD × PTQ — Orchestration Notebook

Single notebook that runs the whole pipeline. Cells below are grouped by phase; each phase calls into the `scripts/` entry points — all real logic lives in `src/` and `scripts/`, not here.

**Typical session**:
1. Run the **Setup** section once per Colab runtime.
2. Run the phase sections you care about (1 → 5).
3. Run the **Commit results** section to push `results/runs/*.json` back to GitHub.

## Setup (run once per runtime)

In [ ]:
# Mount Drive; HF cache & repo live here to survive restarts.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/llm-project/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
# Clone repo to Drive (first run) or pull latest (subsequent runs).
REPO = '/content/drive/MyDrive/llama'
REPO_URL = 'https://github.com/Siyuan3/15442-final.git'

import os, subprocess
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
print('repo at:', REPO)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# HF login. Preferred: add HF_TOKEN via left sidebar 🔑 (Secrets) → Notebook access.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('logged in via Colab Secrets')
except Exception:
    login()

In [ ]:
!nvidia-smi

## Phase 1 — Baseline (Llama-3.2-1B & Llama-3.1-8B, fp16)

Two points on the Pareto frontier: uncompressed student and teacher.

Expected on L4 (23 GB):
| Model | PPL (WikiText-2) | tok/s | VRAM |
|---|---|---|---|
| Llama-3.2-1B fp16 | ~9–10 | ~60–90 | ~2.5 GB |
| Llama-3.1-8B fp16 | ~6–7 | ~15–25 | ~16 GB |

In [ ]:
!python scripts/01_baseline_eval.py --role student

In [ ]:
# If the 8B OOMs on L4, run Phase 2 int8 teacher instead as a proxy baseline.
!python scripts/01_baseline_eval.py --role teacher

## Phase 2 — Post-Training Quantization (bitsandbytes)

Quantize each model to int8 and nf4; compare PPL, tok/s, VRAM vs. Phase 1.

In [ ]:
!python scripts/02_quantize_eval.py --role student --quant int8
!python scripts/02_quantize_eval.py --role student --quant nf4

In [ ]:
!python scripts/02_quantize_eval.py --role teacher --quant int8
!python scripts/02_quantize_eval.py --role teacher --quant nf4

## Phase 3 — Knowledge Distillation (8B → 1B)

⚠ Switch to **A100 (40 GB)** before running this — L4 will OOM with teacher + student + activations + optimizer state.
Training checkpoints go to `checkpoints/distill-llama32-1b/` (configured in `configs/distill.yaml`).

In [ ]:
!python scripts/03_distill_train.py

## Phase 4 — Combined (distilled + quantized) + Pareto plot

In [ ]:
DISTILLED = 'checkpoints/distill-llama32-1b'
!python scripts/04_combined_eval.py eval --path {DISTILLED} --quant none
!python scripts/04_combined_eval.py eval --path {DISTILLED} --quant int8
!python scripts/04_combined_eval.py eval --path {DISTILLED} --quant nf4

In [ ]:
# Aggregate every metrics.json under results/runs/ and render plots/pareto.png.
!python scripts/04_combined_eval.py plot

from IPython.display import Image
Image('plots/pareto.png')

## Phase 5 — Profiling (compute vs memory-movement)

In [ ]:
# Profile the distilled+nf4 config — that's the one we expect to be memory-bound.
!python scripts/05_profile.py --path checkpoints/distill-llama32-1b --quant nf4

## Commit results back to GitHub

`results/runs/*.json` and `plots/` are small — track them in git so the local repo can see them.

In [ ]:
!git config user.email 'colab@local'
!git config user.name 'colab'
!git add results/ plots/
!git commit -m 'results: experiment run from Colab' || echo 'nothing to commit'
!git push